# Confirmation Bias Project
## Behavioural analyses
Launching analysis for each of the experiments: ['3reps', 'CJ']


We recommend to run this script in Jupyter Notebook. It is possible to not visualize some plots in JupyterLab

__Metadprime is computed using the library from:__

pip install git+https://github.com/LegrandNico/metadPy.git

##### Import important functions and libraries

In [ ]:
import os, glob, platform, sys
import numpy as np
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import scipy
import scipy.stats as stats
from scipy import signal
import seaborn as sns
from statsmodels.formula.api import ols
import statsmodels.formula.api as smf
from statsmodels.stats.anova import AnovaRM
#import pingouin as pg

from statsmodels.stats.multicomp import (pairwise_tukeyhsd, MultiComparison)
from matplotlib.lines import Line2D
import statsmodels as sms
#import ptitprince as pt
pd.options.display.max_columns = None # display all the columns in pandas dataframe
#import plotly.graph_objects as go
#import plotly.express as px
#from plotly.subplots import make_subplots



In [ ]:
import pymc
import hddm
pymc.__version__

##### Important functions

__Plotting functions__

In [ ]:
colpal1 = ["orange","green"]
colpal1a = ["deeppink","deepskyblue"]
#colpal2 = ['black','green','blue']
colpal2 = ['darkorange','lime','royalblue']
colpal1b = ['dodgerblue','navy']
colpal3 = ['springgreen','darkviolet']
colpal2 = ['gold','mediumturquoise','darkblue']


In [ ]:
# Importing behavioral data from experiment 
if platform.system() == 'Darwin':
    sys_dir = os.path.join('/Users','alex','Library','CloudStorage','OneDrive-UniversitatdeBarcelona') # macv
    results_path = '/Users/alex/OneDrive - Universitat de Barcelona/Projects/Condcision/'
 
if platform.system() == 'Linux':
    sys_dir = os.path.join('/home/jovyan/work', 'OneDrive Biz')  # Linux   
    results_path = '/home/jovyan/work/OneDrive Biz/PROJECTS/Condcision/' # Linux Mundet

figures_path = os.path.join(results_path, 'Group_level_analyses','figures') 
# some folders with interesting custom functions
path_utils = os.path.join(sys_dir, 'TOOLBOXES','decoding_toolbox_py','Helper_funcs') 
#psychofits = os.path.join(sys_dir, 'TOOLBOXES','psychofit-master')  
sys.path.append(path_utils)
sys.path.append(psychofits)

#import psychofit as psy

Concatenating datasets

Creating individual IDs for each participant (to prevent modelling taking different subjects with similar names as identical)

In [ ]:
# function that I used to plot multiple data
def barsplot(data, dx, dy, hue, col, row, pal, size, yaxis, axislabels, sizepoint, dodge):    
    sns.set(font_scale = 1.5, style = 'ticks')         
    ort = "v"; pal = pal; sigma = .5
    g = sns.FacetGrid(data ,  row = row, col = col, height= size['height'], aspect=size['aspect'], margin_titles=True) # col="nrep",    
    if sizepoint == None:
        sizepoint = 6
    if yaxis != None:
        g.set(yaxis['ylim'], yaxis['yticks'])   

    g.map_dataframe(sns.stripplot, x = dx, y = dy, palette = pal, hue=hue, size = sizepoint, edgecolor = "white",
                    linewidth = 0.6, jitter = 0.2, orient = ort,alpha = 0.5, dodge=dodge)
    g.map_dataframe(sns.barplot, x = dx, y = dy, palette = pal, hue=hue,  linewidth = 0.6, orient = ort, dodge=dodge)
    
    #g.map_dataframe(sns.violinplot, x = dx, y = dy,  palette = pal,bw = .5, cut = 0.,scale = "area", width = .6, inner = None, orient = ort, linewidth = 0, zorder = 2)
    
    g.add_legend()

    sns.despine(offset = .5,  trim=True);
    # Set x-axis and y-axis labels
    g.set_axis_labels( axislabels['xlabel'] , axislabels['ylabel'], fontsize = 15 )
    #g.tight_layout()
    return g


## Drift diffusion analyses

 __HDDM cheat sheet__

 v     = drift rate
 a     = boundary separation
 t     = nondecision time
 z     = starting point
 dc    = drift driterion
 sv    = inter-trial variability in drift-rate
 st    = inter-trial variability in non-decision time
 sz    = inter-trial variability in starting-point


In [ ]:
import hddm

# ============================================ #
# also compute BIC, AIC
# from https://groups.google.com/forum/#!searchin/hddm-users/bic%7Csort:date/hddm-users/Bo2vUcpR008/RLRpL0faptAJ
# ============================================ #
def aic(self):
    k = len(self.get_stochastics())
    logp = sum([x.logp for x in self.get_observeds()['node']])  
    return 2 * k - 2 * logp

def bic(self):
    k = len(self.get_stochastics())
    n = len(self.data)
    logp = sum([x.logp for x in self.get_observeds()['node']])
    return -2 * logp + k * np.log(n)

# model comparison function
def GLRT(mod1, mod2):
    
    chi_square = 2 * abs(mod1.logLike - mod2.logLike)
    delta_params = abs(len(mod1.coefs) - len(mod2.coefs)) 
    
    return {"chi_square" : chi_square, "df": delta_params, "p" : 1 - stats.chi2.cdf(chi_square, df=delta_params)}

# Loading the models

Real data models

In [ ]:

exp_IDs = ['3reps','CJ'] 
#define models ran for each experiment
models_3reps = ['stimcoding_nohist','stimcoding_z_prevresp','stimcoding_dc_prevresp','stimcoding_zXdc_prevresp']
models_CJ = ['stimcoding_nohist','stimcoding_z_prevresp','stimcoding_dc_prevresp','stimcoding_zXdc_prevresp','stimcoding_dc_prevresp_trial_type'] 

# path
hddm_m_path = os.path.join(results_path,'Group_level_analyses','hddm_models')
models = {}

for exp in exp_IDs:
    models[exp] = {}
    
    # change the name of the models as a function of the experiment
    model_name = models_3reps

    if exp == 'CJ':
        model_name = models_CJ
    for mod_name in model_name:
        models[exp][mod_name] = {}
        try:
            model_filename  = os.path.join(hddm_m_path, mod_name+'_'+exp)       
            file = open(model_filename+'_fm.db','rb')
        except:
            print('exception')
        model_file = pickle.load(file)
        file.close()
        models[exp][mod_name] = model_file  #hddm.load(model_filename+'_fm.db')
        
    

__Loading simulated data DDM fits__

path = '/home/jovyan/work/OneDrive Biz/PROJECTS/Condcision/Group_level_analyses/hddm_models/lapse_sim_dc_fm.db'
path = '/home/jovyan/work/OneDrive Biz/PROJECTS/Condcision/Group_level_analyses/hddm_models/threshold_sim_nohist_fm.db'

#path = '/home/jovyan/work/OneDrive Biz/PROJECTS/Condcision/Group_level_analyses/hddm_models/stimcoding_dc_prevresp_trial_type_CJ_fm.db'
file = open(path,'rb')
model_file = pickle.load(file)
file.close()

In [ ]:
exp_IDs = ['threshold_sim', 'lapse_sim', 'Z_sim','slope_sim']  #'3reps_lag',, 
#define models ran for each experiment
models_types = ['nohist','Z','dc']


hddm_m_path = os.path.join(results_path,'Group_level_analyses','hddm_models')
models_sim = {}

for exp in exp_IDs:
    models_sim[exp] = {}
    
    # change the name of the models as a function of the experiment
    for mod_name in models_types:
        models_sim[exp][mod_name] = {}
        try:
            model_filename  = os.path.join(hddm_m_path,  exp +'_'+ mod_name)       
            file = open(model_filename+'_fm.db','rb')
        except:
            print('exception')
        model_file = pickle.load(file)
        file.close()
        models_sim[exp][mod_name] = model_file  #hddm.load(model_filename+'_fm.db')
        
    

In [ ]:
#models_sim['threshold_sim']['Z'].print_stats()

# extracting parameters for plotting


Two functions to get data

In [ ]:
def get_params(models, exp, mod_name):
    coef = models.gen_stats()
    if exp == '3reps':
        if mod_name == 'stimcoding_nohist':
            print(mod_name)
            v = coef.loc[['v(0)','v(1)','v(2)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1,2])
            v['prevr'] = np.array(['N','N','N'])
            v['repeated'] = np.array(['N','N','N'])
            v['lag'] = np.array(['N','N','N'])
            
            z = coef.loc[['z(0)','z(1)','z(2)'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,1,2])
            z['prevr'] = np.array(['N','N','N'])
            z['repeated'] = np.array(['N','N','N'])
            z['lag'] = np.array(['N','N','N'])


            dat = pd.concat([v,z]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)
            
        if mod_name == 'stimcoding_z_prevresp':
            print(mod_name)
            v = coef.loc[['v(0)','v(1)','v(2)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1,2])
            v['prevr'] = np.array(['N','N','N'])
            v['repeated'] = np.array(['N','N','N'])
            v['lag'] = np.array(['N','N','N'])
            
            z = coef.loc[['z(0.0)','z(0.1)','z(1.0)','z(1.1)','z(2.0)','z(2.1)'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,0,1,1,2,2])
            z['prevr'] = np.array(['C','D','C','D','C','D'])
            z['repeated'] = np.array(['N','N','N','N','N','N'])
            z['lag'] = np.array(['N','N','N','N','N','N'])
            

            dat = pd.concat([v,z]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)
            
        
        if mod_name == 'stimcoding_dc_prevresp':
            print(mod_name)
            v = coef.loc[['v(0)','v(1)','v(2)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1,2])
            v['prevr'] = np.array(['N','N','N'])
            v['repeated'] = np.array(['N','N','N'])
            v['lag'] = np.array(['N','N','N'])
            
            
            z = coef.loc[['z','z','z'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,1,2])
            z['prevr'] = np.array(['N','N','N'])
            z['repeated'] = np.array(['N','N','N'])
            z['lag'] = np.array(['N','N','N'])
            
            dc = coef.loc[['dc(0.0)','dc(0.1)','dc(1.0)','dc(1.1)','dc(2.0)','dc(2.1)'], ['mean','2.5q','97.5q']]
            dc['param'] = 'dc'
            dc['nrep'] = np.array([0,0,1,1,2,2])
            dc['prevr'] = np.array(['C','D','C','D','C','D'])
            dc['repeated'] = np.array(['N','N','N','N','N','N'])
            dc['lag'] = np.array(['N','N','N','N','N','N'])

            dat = pd.concat([v,z,dc]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)
            
        if mod_name == 'stimcoding_zXdc_prevresp':
            print(mod_name)
            v = coef.loc[['v(0)','v(1)','v(2)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1,2])
            v['prevr'] = np.array(['N','N','N'])
            v['repeated'] = np.array(['N','N','N'])
            v['lag'] = np.array(['N','N','N'])
            
            z = coef.loc[['z(0.0)','z(0.1)','z(1.0)','z(1.1)','z(2.0)','z(2.1)'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,0,1,1,2,2])
            z['prevr'] = np.array(['C','D','C','D','C','D'])
            z['repeated'] = np.array(['N','N','N','N','N','N'])
            z['lag'] = np.array(['N','N','N','N','N','N'])
            
            dc = coef.loc[['dc(0.0)','dc(0.1)','dc(1.0)','dc(1.1)','dc(2.0)','dc(2.1)'], ['mean','2.5q','97.5q']]
            dc['param'] = 'dc'
            dc['nrep'] = np.array([0,0,1,1,2,2])
            dc['prevr'] = np.array(['C','D','C','D','C','D'])
            dc['repeated'] = np.array(['N','N','N','N','N','N'])
            dc['lag'] = np.array(['N','N','N','N','N','N'])

            dat = pd.concat([v,z,dc]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)

    # change name of repetitions
    if exp == 'CJ':
        if mod_name == 'stimcoding_nohist':
            print(mod_name)
            v = coef.loc[['v(0)','v(1)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1])
            v['prevr'] = np.array(['N','N'])
            v['repeated'] = np.array(['N','N'])
            v['lag'] = np.array(['N','N'])
            
            z = coef.loc[['z','z'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,1])
            z['prevr'] = np.array(['N','N'])
            z['repeated'] = np.array(['N','N'])
            z['lag'] = np.array(['N','N'])


            dat = pd.concat([v,z]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)

        if mod_name == 'stimcoding_z_prevresp':
            print(mod_name)
            v = coef.loc[['v(0)','v(1)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1])
            v['prevr'] = np.array(['N','N'])
            v['repeated'] = np.array(['N','N'])
            v['lag'] = np.array(['N','N'])
            
            z = coef.loc[['z(0.0)','z(0.1)','z(1.0)','z(1.1)'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,0,1,1])
            z['prevr'] = np.array(['C','D','C','D'])
            z['repeated'] = np.array(['N','N','N','N'])
            z['lag'] = np.array(['N','N','N','N'])
            

            dat = pd.concat([v,z]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)
            
        
        if mod_name == 'stimcoding_dc_prevresp':
            print(mod_name)
            v = coef.loc[['v(0)','v(1)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1])
            v['prevr'] = np.array(['N','N'])
            v['repeated'] = np.array(['N','N'])
            v['lag'] = np.array(['N','N'])
            
            
            z = coef.loc[['z','z'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,1])
            z['prevr'] = np.array(['N','N'])
            z['repeated'] = np.array(['N','N'])
            z['lag'] = np.array(['N','N'])
            
            dc = coef.loc[['dc(0.0)','dc(0.1)','dc(1.0)','dc(1.1)'], ['mean','2.5q','97.5q']]
            dc['param'] = 'dc'
            dc['nrep'] = np.array([0,0,1,1])
            dc['prevr'] = np.array(['C','D','C','D'])
            dc['repeated'] = np.array(['N','N','N','N'])
            dc['lag'] = np.array(['N','N','N','N'])

            dat = pd.concat([v,z,dc]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)
            
        if mod_name == 'stimcoding_zXdc_prevresp':
            print(mod_name)
            v = coef.loc[['v(0)','v(1)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1])
            v['prevr'] = np.array(['N','N'])
            v['repeated'] = np.array(['N','N'])
            v['lag'] = np.array(['N','N'])
            
            z = coef.loc[['z(0.0)','z(0.1)','z(1.0)','z(1.1)'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,0,1,1])
            z['prevr'] = np.array(['C','D','C','D'])
            z['repeated'] = np.array(['N','N','N','N'])
            z['lag'] = np.array(['N','N','N','N'])
            
            dc = coef.loc[['dc(0.0)','dc(0.1)','dc(1.0)','dc(1.1)'], ['mean','2.5q','97.5q']]
            dc['param'] = 'dc'
            dc['nrep'] = np.array([0,0,1,1])
            dc['prevr'] = np.array(['C','D','C','D'])
            dc['repeated'] = np.array(['N','N','N','N'])
            dc['lag'] = np.array(['N','N','N','N'])
            
            
            dat = pd.concat([v,z,dc]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)
            
        if mod_name == 'stimcoding_dc_prevresp_trial_type':
            print(mod_name)
            v = coef.loc[['v(0.nonrepeat)','v(0.repeat)','v(1.nonrepeat)','v(1.repeat)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,0,1,1])
            v['prevr'] = np.array(['N','N','N','N'])
            v['repeated'] = np.array(['NR','R','NR','R'])
            v['lag'] = np.array(['N','N','N','N'])

            z = coef.loc[['z','z'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,1])
            z['prevr'] = np.array(['N','N'])
            z['repeated'] = np.array(['N','N'])
            z['lag'] = np.array(['N','N'])

            dc = coef.loc[['dc(0.0.nonrepeat)','dc(0.0.repeat)','dc(0.1.nonrepeat)','dc(0.1.repeat)','dc(1.0.nonrepeat)','dc(1.0.repeat)','dc(1.1.nonrepeat)','dc(1.1.repeat)'], ['mean','2.5q','97.5q']]
            dc['param'] = 'dc'
            dc['nrep'] = np.array([0,0,0,0,1,1,1,1])
            dc['prevr'] = np.array(['C','C','D','D','C','C','D','D'])
            dc['repeated'] = np.array(['NR','R','NR','R','NR','R','NR','R'])
            dc['lag'] = np.array(['N','N','N','N','N','N','N','N'])
            
            
            dat = pd.concat([v,z,dc]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)
   
            
    

        if mod_name == 'stimcoding_dc_prevresp':
            print(mod_name)
            v = coef.loc[['v(250.0.0.0)','v(250.0.1.0)','v(250.0.2.0)','v(3500.0.0.0)','v(3500.0.1.0)','v(3500.0.2.0)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1,2,0,1,2])
            v['prevr'] = np.array(['N','N','N','N','N','N'])
            v['repeated'] = np.array(['N','N','N','N','N','N'])
            v['lag'] = np.array(['250','250','250','3500','3500','3500'])
            
            
            z = coef.loc[['z','z','z'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,1,2])
            z['prevr'] = np.array(['N','N','N'])
            z['repeated'] = np.array(['N','N','N'])
            z['lag'] = np.array(['N','N','N'])
            
            dc = coef.loc[['dc(250.0.0.0.0.0)','dc(250.0.0.0.1.0)','dc(250.0.1.0.0.0)','dc(250.0.1.0.1.0)','dc(250.0.2.0.0.0)','dc(250.0.2.0.1.0)', 'dc(3500.0.0.0.0.0)','dc(3500.0.0.0.1.0)','dc(3500.0.1.0.0.0)','dc(3500.0.1.0.1.0)','dc(3500.0.2.0.0.0)','dc(3500.0.2.0.1.0)'], ['mean','2.5q','97.5q']]
            dc['param'] = 'dc'
            dc['nrep'] = np.array([0,0,1,1,2,2,0,0,1,1,2,2])
            dc['prevr'] = np.array(['C','D','C','D','C','D','C','D','C','D','C','D'])
            dc['repeated'] = np.array(['N','N','N','N','N','N','N','N','N','N','N','N'])
            dc['lag'] = np.array(['250','250','250','250','250','250','3500','3500','3500','3500','3500','3500'])

            dat = pd.concat([v,z,dc]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)
            
        if mod_name == 'stimcoding_zXdc_prevresp':
            print(mod_name)
            v = coef.loc[['v(250.0.0.0)','v(250.0.1.0)','v(250.0.2.0)','v(3500.0.0.0)','v(3500.0.1.0)','v(3500.0.2.0)'], ['mean','2.5q','97.5q']]
            v['param'] = 'v'
            v['nrep'] = np.array([0,1,2,0,1,2])
            v['prevr'] = np.array(['N','N','N','N','N','N'])
            v['repeated'] = np.array(['N','N','N','N','N','N'])
            v['lag'] = np.array(['250','250','250','3500','3500','3500'])
            
            z = coef.loc[['z(250.0.0.0.0.0)','z(250.0.0.0.1.0)','z(250.0.1.0.0.0)','z(250.0.1.0.1.0)','z(250.0.2.0.0.0)','z(250.0.2.0.1.0)', 'z(3500.0.0.0.0.0)','z(3500.0.0.0.1.0)','z(3500.0.1.0.0.0)','z(3500.0.1.0.1.0)','z(3500.0.2.0.0.0)','z(3500.0.2.0.1.0)'], ['mean','2.5q','97.5q']]
            z['param'] = 'z'
            z['nrep'] = np.array([0,0,1,1,2,2,0,0,1,1,2,2])
            z['prevr'] = np.array(['C','D','C','D','C','D','C','D','C','D','C','D'])
            z['repeated'] = np.array(['N','N','N','N','N','N','N','N','N','N','N','N'])
            z['lag'] = np.array(['250','250','250','250','250','250','3500','3500','3500','3500','3500','3500'])
            
            dc = coef.loc[['dc(250.0.0.0.0.0)','dc(250.0.0.0.1.0)','dc(250.0.1.0.0.0)','dc(250.0.1.0.1.0)','dc(250.0.2.0.0.0)','dc(250.0.2.0.1.0)', 'dc(3500.0.0.0.0.0)','dc(3500.0.0.0.1.0)','dc(3500.0.1.0.0.0)','dc(3500.0.1.0.1.0)','dc(3500.0.2.0.0.0)','dc(3500.0.2.0.1.0)'], ['mean','2.5q','97.5q']]
            dc['param'] = 'dc'
            dc['nrep'] = np.array([0,0,1,1,2,2,0,0,1,1,2,2])
            dc['prevr'] = np.array(['C','D','C','D','C','D','C','D','C','D','C','D'])
            dc['repeated'] = np.array(['N','N','N','N','N','N','N','N','N','N','N','N'])
            dc['lag'] = np.array(['250','250','250','250','250','250','3500','3500','3500','3500','3500','3500'])
            
            
            dat = pd.concat([v,z,dc]).reset_index()
            dat['exp_ID'] = exp
            dat['model'] = mod_name 
            dat['aic'] = aic(models)
            dat['bic'] = bic(models)
             
        
    return dat

# models from simulate data
def get_simparams(models, exp, mod_name):
    coef = models.gen_stats()    
    if mod_name == 'nohist':
        print(mod_name)
        v = coef.loc[['v(0.0)','v(1.0)'], ['mean','2.5q','97.5q']]
        v['param'] = 'v'
        v['nrep'] = np.array([0,1])
        v['prevr'] = np.array(['N','N'])
        v['repeated'] = np.array(['N','N'])
        v['lag'] = np.array(['N','N'])

        z = coef.loc[['z'], ['mean','2.5q','97.5q']]
        z['param'] = 'z'
        z['nrep'] = np.array(['N'])
        z['prevr'] = np.array(['N'])
        z['repeated'] = np.array(['N'])
        z['lag'] = np.array(['N'])


        dat = pd.concat([v,z]).reset_index()
        dat['exp_ID'] = exp
        dat['model'] = mod_name 
        dat['aic'] = aic(models)
        dat['bic'] = bic(models)

    if mod_name == 'Z':
        print(mod_name)
        v = coef.loc[['v(0.0)','v(1.0)'], ['mean','2.5q','97.5q']]
        v['param'] = 'v'
        v['nrep'] = np.array([0,1])
        v['prevr'] = np.array(['N','N'])
        v['repeated'] = np.array(['N','N'])
        v['lag'] = np.array(['N','N'])

        z = coef.loc[['z(0.0.0.0)','z(0.0.1.0)','z(1.0.0.0)','z(1.0.1.0)'], ['mean','2.5q','97.5q']]
        z['param'] = 'z'
        z['nrep'] = np.array([0,0,1,1])
        z['prevr'] = np.array(['C','D','C','D'])
        z['repeated'] = np.array(['N','N','N','N'])
        z['lag'] = np.array(['N','N','N','N'])


        dat = pd.concat([v,z]).reset_index()
        dat['exp_ID'] = exp
        dat['model'] = mod_name 
        dat['aic'] = aic(models)
        dat['bic'] = bic(models)


    if mod_name == 'dc':
        print(mod_name)
        v = coef.loc[['v(0.0)','v(1.0)'], ['mean','2.5q','97.5q']]
        v['param'] = 'v'
        v['nrep'] = np.array([0,1])
        v['prevr'] = np.array(['N','N'])
        v['repeated'] = np.array(['N','N'])
        v['lag'] = np.array(['N','N'])


        z = coef.loc[['z'], ['mean','2.5q','97.5q']]
        z['param'] = 'z'
        z['nrep'] = np.array(['N'])
        z['prevr'] = np.array(['N'])
        z['repeated'] = np.array(['N'])
        z['lag'] = np.array(['N'])

        dc = coef.loc[['dc(0.0.0.0)','dc(0.0.1.0)','dc(1.0.0.0)','dc(1.0.1.0)'], ['mean','2.5q','97.5q']]
        dc['param'] = 'dc'
        dc['nrep'] = np.array([0,0,1,1])
        dc['prevr'] = np.array(['C','D','C','D'])
        dc['repeated'] = np.array(['N','N','N','N'])
        dc['lag'] = np.array(['N','N','N','N'])

        dat = pd.concat([v,z,dc]).reset_index()
        dat['exp_ID'] = exp
        dat['model'] = mod_name 
        dat['aic'] = aic(models)
        dat['bic'] = bic(models)

    return dat

Function to extract the DDM parameters of real data

In [ ]:
exp_IDs = ['3reps','CJ']  #'3reps_lag',
#define models ran for each experiment
models_3reps = ['stimcoding_nohist','stimcoding_z_prevresp','stimcoding_dc_prevresp','stimcoding_zXdc_prevresp']
models_CJ = ['stimcoding_nohist','stimcoding_z_prevresp','stimcoding_dc_prevresp','stimcoding_zXdc_prevresp','stimcoding_dc_prevresp_trial_type''stimcoding_diff_prev_repeat','stimcoding_diff_prev_nonrepeat'] 

i = 0
for im in models:
    for mod_name in models[im]:
        if i == 0:
            params = get_params( models[im][mod_name], im, mod_name)

            i = i + 1
        else:
            aux = get_params(models[im][mod_name], im, mod_name)
            #print(aux)
            params = pd.concat([params, aux])
        
params.reset_index(inplace = True, drop = True)       

In [ ]:
#del simparams
exp_IDs = ['threshold_sim', 'lapse_sim','Z_sim','slope_sim']
#define models ran for each experiment
models_types = ['nohist','Z','dc']


i = 0

for im in exp_IDs:
    for mod_name in models_sim[im]:
        if i == 0:
            simparams = get_simparams( models_sim[im][mod_name], im, mod_name)

            i = i + 1
        else:
            aux = get_simparams(models_sim[im][mod_name], im, mod_name)
            #print(aux)
            simparams = pd.concat([simparams, aux])
        
simparams.reset_index(inplace = True, drop = True)       

In [ ]:
all_params = pd.concat([params,simparams])
all_params.reset_index(inplace=True, drop=True)

In [ ]:
all_params.exp_ID.unique()

In [ ]:
params_path = os.path.join(hddm_m_path,  'DDMs_params.csv')  
all_params.to_csv(params_path )

# HDDM Stimuli regression parameters

In [ ]:
hddm_m_path = os.path.join(results_path,'Group_level_analyses','hddm_models')
models_reg = {}

exp_IDs = ['3reps','CJ']
model_namesall = ['stimcoding_weights','stimcoding_diffprev']
model_namesCJ = ['stimcoding_weights','stimcoding_diffprev','stimcoding_diff_prev_repeat','stimcoding_diff_prev_nonrepeat']

for exp in exp_IDs:
    models_reg[exp] = {}
    
    model_name = model_namesall    
    if exp == 'CJ':
        model_name = model_namesCJ 
    
    for mod_name in model_name:
        models_reg[exp][mod_name] = {}
        try:
            model_filename  = os.path.join(hddm_m_path, mod_name+'_'+exp)       
            file = open(model_filename+'_fm.db','rb')
        except:
            print('exception')
            
        model_file = pickle.load(file)
        file.close()
        models_reg[exp][mod_name] = model_file  #hddm.load(model_filename+'_fm.db')
        

hddm_m_path = os.path.join(results_path,'Group_level_analyses','hddm_models')
models_stim = {}
exp = '3reps'
mod_name = 'stimcoding_diffprev'#'stimcoding_weights'
models_stim[exp] = {}
    
try:
    model_filename  = os.path.join(hddm_m_path, mod_name+'_'+exp)       
    file = open(model_filename+'_fm.db','rb')
    model_file = pickle.load(file)
except:
    print('exception')
file.close()
models_stim[exp][mod_name] = model_file  #hddm.load(model_filename+'_fm.db')

In [ ]:
coef = models_reg['CJ']['stimcoding_diff_prev_nonrepeat'].gen_stats()

In [ ]:
mod_name = 'stimcoding_diff_prev_nonrepeat'

In [ ]:
mod_name.startswith('stimcoding_diff')

In [ ]:
def get_reg_params(models, exp, mod_name):
    coef = models[exp][mod_name].gen_stats()
    if mod_name.startswith('stimcoding_diff'):
        z = coef.loc[['z_Intercept','z_response1','z_C(nrep)[T.1]','z_response1:C(nrep)[T.1]'], ['mean','2.5q','97.5q']]
        base_val = z.iloc[0,0]
        z['nrep'] = np.array([0,0,1,1])
        z['prevr'] = np.array([0,1,0,1])
        z['pos'] = 'N'
        z['param'] = 'z'
        z['2.5q'] = z['2.5q'] - z['mean']  
        z['97.5q'] = z['97.5q'] - z['mean']  
        z['mean'] = z['mean']  + base_val
        z.iloc[0,0] = z.iloc[0,0] - base_val
        
        if exp != 'CJ':        
            z = coef.loc[['z_Intercept','z_response1','z_C(nrep)[T.1]','z_response1:C(nrep)[T.1]','z_C(nrep)[T.2]','z_response1:C(nrep)[T.2]'], ['mean','2.5q','97.5q']]
            base_val = z.iloc[0,0]
            z['nrep'] = np.array([0,0,1,1,2,2])
            z['prevr'] = np.array([0,1,0,1,0,1])
            z['pos'] = 'N'
            z['param'] = 'z'
            z['2.5q'] = z['2.5q'] - z['mean']  
            z['97.5q'] = z['97.5q'] - z['mean']  
            z['mean'] = z['mean']  + base_val
            z.iloc[0,0] = z.iloc[0,0] - base_val


        #v_Intercept
        base_val = coef.loc['v_Intercept']['mean']

        v0 = coef.loc[['v_G1sim:C(nrep)[0]','v_G2sim:C(nrep)[0]','v_G3sim:C(nrep)[0]','v_G4sim:C(nrep)[0]','v_G5sim:C(nrep)[0]','v_G6sim:C(nrep)[0]'], ['mean','2.5q','97.5q']]
        v0['nrep'] = 0
        v0['prevr'] = 'N'
        v0['pos'] = np.array([1,2,3,4,5,6])
        v0['param'] = 'v'
        v0['2.5q'] = v0['2.5q'] - v0['mean']  
        v0['97.5q'] = v0['97.5q'] - v0['mean']  
        v0['mean']  = v0['mean']  + base_val 

        v1 = coef.loc[['v_G1sim:C(nrep)[1]','v_G2sim:C(nrep)[1]','v_G3sim:C(nrep)[1]','v_G4sim:C(nrep)[1]','v_G5sim:C(nrep)[1]','v_G6sim:C(nrep)[1]'], ['mean','2.5q','97.5q']]
        v1['nrep'] = 1
        v1['prevr'] = 'N'
        v1['pos'] = np.array([1,2,3,4,5,6])
        v1['param'] = 'v'
        v1['2.5q'] = v1['2.5q'] - v1['mean']  
        v1['97.5q'] = v1['97.5q'] - v1['mean']
        v1['mean']  = v1['mean']  + base_val 
        pars = pd.concat([z,v0,v1])
        pars['model'] = mod_name
        pars['exp'] = exp

        if exp != 'CJ':

            v2 = coef.loc[['v_G1sim:C(nrep)[2]','v_G2sim:C(nrep)[2]','v_G3sim:C(nrep)[2]','v_G4sim:C(nrep)[2]','v_G5sim:C(nrep)[2]','v_G6sim:C(nrep)[2]'], ['mean','2.5q','97.5q']]
            v2['nrep'] = 2
            v2['prevr'] = 'N'
            v2['pos'] = np.array([1,2,3,4,5,6])
            v2['param'] = 'v'
            v2['2.5q'] = v2['2.5q'] - v2['mean']  
            v2['97.5q'] = v2['97.5q'] - v2['mean']
            v2['mean']  = v2['mean']  + base_val 
            pars = pd.concat([z,v0,v1,v2])
            pars['model'] = mod_name
            pars['exp'] = exp
            
            
        return(pars)




concatenating parameters for plotting

In [ ]:
i = 0
for im in models_reg:
    
      #model_name = model_namesall 
    for mod_name in models_reg[im]:
        if i == 0:
            print(mod_name)
            params_reg = get_reg_params( models_reg, im, mod_name)

            i = i + 1
        else:
            print(mod_name)
            aux = get_reg_params(models_reg, im, mod_name)
            #print(aux)
            params_reg = pd.concat([params_reg, aux])
        
params_reg.reset_index(inplace = True, drop = True)     

In [ ]:
params_path = os.path.join(hddm_m_path,  'DDMs_params_reg.csv')  
params_reg.to_csv(params_path )

In [ ]:

#estims = md_3reps_betas.copy()
fig, axes = plt.subplots(3, 2, figsize=(9,8), gridspec_kw={'width_ratios': [2, 1]})
fig.tight_layout(pad=1.0)

y_limits = np.array([[-0.35, 1.75] ,[0.35, 0.6]])
estims = params_reg[(params_reg.exp == '3reps' ) & (params_reg.model == 'stimcoding_diffprev')]
estims['mean'] = estims['mean'].astype(float)
estims['2.5q'] = estims['2.5q'].astype(float)
estims['97.5q'] = estims['97.5q'].astype(float)
parameters = ['v','z']
nreps = [0,1,2]


for x, xparam in enumerate(parameters):
    for y, yrep in enumerate(nreps):
        if xparam == 'v':
            stim = estims[(estims.param == xparam) & (estims.nrep == yrep)]
            g = sns.pointplot(data=stim, x="pos" , y="mean" , linestyles='', 
            dodge=True, join=False, ci=None,scale=1.5,  palette = "magma", errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[y,x])
            
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[y, x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords)      
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs = np.abs(errs)
            axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            fmt=' ', zorder=3, ecolor = 'black')
            
            axes[y,x].set_ylim(y_limits[0])
            axes[y,x].set_xlim([-1,6])
            
            axes[y,x].tick_params(axis='y', labelsize=15) 
            axes[y,x].tick_params(axis='x', labelsize=15)
            axes[y,x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[y,x].spines['top'].set_visible(False)
            axes[y,x].spines['right'].set_visible(False)
            axes[y,x].spines['left'].set_visible(False)
            
            axes[y,x].set_xlabel('Similarity N-1', fontsize = 15)
            axes[y,x].set_ylabel('Estimate (a.u.)', fontdict={'size':15}); 
            axes[y,x].margins(x=0.5)
            axes[y,x].set_xticklabels(['1st','2nd','3rd','4th','5th','6th'])


        if xparam == 'z':
            stim = estims[(estims.param == xparam) & (estims.nrep == yrep)]
            g = sns.pointplot(data=stim, x="prevr" , y="mean" , hue="prevr",linestyles='', 
            dodge=True, join=False, ci=None,scale=1.5, palette = colpal3, errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[y,x])
            
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[y, x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords)      
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs = np.abs(errs)

            axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            fmt=' ', zorder=3, ecolor = 'black')
            
            axes[y,x].set_ylim(y_limits[1])
            axes[y,x].set_xlim([-0.5,1.5])
            axes[y,x].tick_params(axis='y', labelsize=15) 
            axes[y,x].tick_params(axis='x', labelsize=15) 
            axes[y,x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[y,x].spines['top'].set_visible(False)
            axes[y,x].spines['right'].set_visible(False)
            axes[y,x].spines['left'].set_visible(False)
            axes[y,x].set_xlabel('Choice N-1', fontsize = 15)
            axes[y,x].set_ylabel('Estimate (a.u.)', fontdict={'size':15}); 
            axes[y,x].margins(x=0.5)
            axes[y,x].set_xticklabels(['C', 'D'])
            axes[y,x].set_yticks([0.4,0.5,0.6])
            axes[y,x].set_yticklabels([0.4,0.5,0.6])

            #axes[y,x].margins(y=10)



figpath = os.path.join(figures_path, '3reps_hddmREG.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)
     

In [ ]:

#estims = md_3reps_betas.copy()
fig, axes = plt.subplots(2, 2, figsize=(10,7), gridspec_kw={'width_ratios': [2, 1]})
fig.tight_layout(pad=2.0)

y_limits = np.array([[-0.35, 1.75] ,[0.35, 0.6]])
estims = params_reg[(params_reg.exp == 'CJ' ) & (params_reg.model == 'stimcoding_diff_prev_repeat')]
estims['mean'] = estims['mean'].astype(float)
estims['2.5q'] = estims['2.5q'].astype(float)
estims['97.5q'] = estims['97.5q'].astype(float)
parameters = ['v','z']
nreps = [0,1]


for x, xparam in enumerate(parameters):
    for y, yrep in enumerate(nreps):
        if xparam == 'v':
            stim = estims[(estims.param == xparam) & (estims.nrep == yrep)]
            g = sns.pointplot(data=stim, x="pos" , y="mean" , linestyles='', 
            dodge=True, join=False, ci=None,scale=1.5,  palette = "magma", errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[y,x])
            
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[y, x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords)      
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs = np.abs(errs)
            axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            fmt=' ', zorder=3, ecolor = 'black')
            
            axes[y,x].set_ylim(y_limits[0])
            axes[y,x].set_xlim([-1,6])
            
            axes[y,x].tick_params(axis='y', labelsize=15) 
            axes[y,x].tick_params(axis='x', labelsize=15)
            axes[y,x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[y,x].spines['top'].set_visible(False)
            axes[y,x].spines['right'].set_visible(False)
            axes[y,x].spines['left'].set_visible(False)
            
            axes[y,x].set_xlabel('Similarity N-1', fontsize = 15)
            axes[y,x].set_ylabel('v estimate (a.u.)', fontdict={'size':15}); 
            axes[y,x].margins(x=0.5)
            axes[y,x].set_xticklabels(['1st','2nd','3rd','4th','5th','6th'])


        if xparam == 'z':
            stim = estims[(estims.param == xparam) & (estims.nrep == yrep)]
            g = sns.pointplot(data=stim, x="prevr" , y="mean" , hue="prevr",linestyles='', 
            dodge=True, join=False, ci=None,scale=1.5, palette = colpal3, errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[y,x])
            
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[y, x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords)      
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs = np.abs(errs)

            axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            fmt=' ', zorder=3, ecolor = 'black')
            
            axes[y,x].set_ylim(y_limits[1])
            axes[y,x].set_xlim([-0.5,1.5])
            axes[y,x].tick_params(axis='y', labelsize=15) 
            axes[y,x].tick_params(axis='x', labelsize=15) 
            axes[y,x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[y,x].spines['top'].set_visible(False)
            axes[y,x].spines['right'].set_visible(False)
            axes[y,x].spines['left'].set_visible(False)
            axes[y,x].set_xlabel('Choice N-1', fontsize = 15)
            axes[y,x].set_ylabel('z estimate (a.u.)', fontdict={'size':15}); 
            axes[y,x].margins(x=0.5)
            axes[y,x].set_xticklabels(['C', 'D'])
            axes[y,x].set_yticks([0.4,0.5,0.6])
            axes[y,x].set_yticklabels([0.4,0.5,0.6])

            #axes[y,x].margins(y=10)


 
            
           # sns.lineplot(data=stim, x="prevr" , y="mean" , ax=axes[y,x])



figpath = os.path.join(figures_path, 'CJ_hddmREG.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)

        

In [ ]:

#estims = md_3reps_betas.copy()
fig, axes = plt.subplots(2, 2, figsize=(10,7), gridspec_kw={'width_ratios': [2, 1]})
fig.tight_layout(pad=2.0)

y_limits = np.array([[-0.35, 1.75] ,[0.35, 0.6]])
estims = params_reg[(params_reg.exp == 'CJ' ) & (params_reg.model == 'stimcoding_diff_prev_nonrepeat')]
estims['mean'] = estims['mean'].astype(float)
estims['2.5q'] = estims['2.5q'].astype(float)
estims['97.5q'] = estims['97.5q'].astype(float)
parameters = ['v','z']
nreps = [0,1]


for x, xparam in enumerate(parameters):
    for y, yrep in enumerate(nreps):
        if xparam == 'v':
            stim = estims[(estims.param == xparam) & (estims.nrep == yrep)]
            g = sns.pointplot(data=stim, x="pos" , y="mean" , linestyles='', 
            dodge=True, join=False, ci=None,scale=1.5,  palette = "magma", errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[y,x])
            
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[y, x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords)      
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs = np.abs(errs)
            axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            fmt=' ', zorder=3, ecolor = 'black')
            
            axes[y,x].set_ylim(y_limits[0])
            axes[y,x].set_xlim([-1,6])
            
            axes[y,x].tick_params(axis='y', labelsize=15) 
            axes[y,x].tick_params(axis='x', labelsize=15)
            axes[y,x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[y,x].spines['top'].set_visible(False)
            axes[y,x].spines['right'].set_visible(False)
            axes[y,x].spines['left'].set_visible(False)
            
            axes[y,x].set_xlabel('Similarity N-1', fontsize = 15)
            axes[y,x].set_ylabel('v estimate (a.u.)', fontdict={'size':15}); 
            axes[y,x].margins(x=0.5)
            axes[y,x].set_xticklabels(['1st','2nd','3rd','4th','5th','6th'])


        if xparam == 'z':
            stim = estims[(estims.param == xparam) & (estims.nrep == yrep)]
            g = sns.pointplot(data=stim, x="prevr" , y="mean" , hue="prevr",linestyles='', 
            dodge=True, join=False, ci=None,scale=1.5, palette = colpal3, errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[y,x])
            
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[y, x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords)      
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs = np.abs(errs)

            axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            fmt=' ', zorder=3, ecolor = 'black')
            
            axes[y,x].set_ylim(y_limits[1])
            axes[y,x].set_xlim([-0.5,1.5])
            axes[y,x].tick_params(axis='y', labelsize=15) 
            axes[y,x].tick_params(axis='x', labelsize=15) 
            axes[y,x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[y,x].spines['top'].set_visible(False)
            axes[y,x].spines['right'].set_visible(False)
            axes[y,x].spines['left'].set_visible(False)
            axes[y,x].set_xlabel('Choice N-1', fontsize = 15)
            axes[y,x].set_ylabel('z estimate (a.u.)', fontdict={'size':15}); 
            axes[y,x].margins(x=0.5)
            axes[y,x].set_xticklabels(['C', 'D'])
            axes[y,x].set_yticks([0.4,0.5,0.6])
            axes[y,x].set_yticklabels([0.4,0.5,0.6])

            #axes[y,x].margins(y=10)


 
            
           # sns.lineplot(data=stim, x="prevr" , y="mean" , ax=axes[y,x])



figpath = os.path.join(figures_path, 'CJ_hddmREG.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)

        

In [ ]:

#estims = md_3reps_betas.copy()
fig, axes = plt.subplots(3, 2, figsize=(10,9), gridspec_kw={'width_ratios': [2, 1]})
fig.tight_layout(pad=2.0)

y_limits = np.array([[-0.35, 1.75] ,[0.35, 0.6]])
# DEPRECATED: estims = params_reg[(params_reg.exp == '3reps_lag' ) & (params_reg.model == 'stimcoding_diffprev')]
estims['mean'] = estims['mean'].astype(float)
estims['2.5q'] = estims['2.5q'].astype(float)
estims['97.5q'] = estims['97.5q'].astype(float)
parameters = ['v','z']
nreps = [0,1,2]


for x, xparam in enumerate(parameters):
    for y, yrep in enumerate(nreps):
        if xparam == 'v':
            stim = estims[(estims.param == xparam) & (estims.nrep == yrep)]
            g = sns.pointplot(data=stim, x="pos" , y="mean" , linestyles='', 
            dodge=True, join=False, ci=None,scale=1.5,  palette = "magma", errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[y,x])
            
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[y, x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords)      
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs = np.abs(errs)
            axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            fmt=' ', zorder=3, ecolor = 'black')
            
            axes[y,x].set_ylim(y_limits[0])
            axes[y,x].set_xlim([-1,6])
            
            axes[y,x].tick_params(axis='y', labelsize=15) 
            axes[y,x].tick_params(axis='x', labelsize=15)
            axes[y,x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[y,x].spines['top'].set_visible(False)
            axes[y,x].spines['right'].set_visible(False)
            axes[y,x].spines['left'].set_visible(False)
            
            axes[y,x].set_xlabel('Similarity N-1', fontsize = 15)
            axes[y,x].set_ylabel('Estimate (a.u.)', fontdict={'size':15}); 
            axes[y,x].margins(x=0.5)
            axes[y,x].set_xticklabels(['1st','2nd','3rd','4th','5th','6th'])


        if xparam == 'z':
            stim = estims[(estims.param == xparam) & (estims.nrep == yrep)]
            g = sns.pointplot(data=stim, x="prevr" , y="mean" , hue="prevr",linestyles='', 
            dodge=True, join=False, ci=None,scale=1.5, palette = colpal3, errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[y,x])
            
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[y, x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords)      
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs = np.abs(errs)

            axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            fmt=' ', zorder=3, ecolor = 'black')
            
            axes[y,x].set_ylim(y_limits[1])
            axes[y,x].set_xlim([-0.5,1.5])
            axes[y,x].tick_params(axis='y', labelsize=15) 
            axes[y,x].tick_params(axis='x', labelsize=15) 
            axes[y,x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[y,x].spines['top'].set_visible(False)
            axes[y,x].spines['right'].set_visible(False)
            axes[y,x].spines['left'].set_visible(False)
            axes[y,x].set_xlabel('Choice N-1', fontsize = 15)
            axes[y,x].set_ylabel('Estimate (a.u.)', fontdict={'size':15}); 
            axes[y,x].margins(x=0.5)
            axes[y,x].set_xticklabels(['C', 'D'])
            axes[y,x].set_yticks([0.4,0.5,0.6])
            axes[y,x].set_yticklabels([0.4,0.5,0.6])

          

# Models with interactions with other variables

In [ ]:
model_filename  = os.path.join(hddm_m_path,'z_v_prevrespconf'+'_'+exp)       
file = open(model_filename+'_fm.db','rb')
model_file = pickle.load(file)
file.close()

In [ ]:
os.path.join(hddm_m_path, mod_name+'_'+exp) 

In [ ]:
hddm_m_path = os.path.join(results_path,'Group_level_analyses','hddm_models')
models_reg_CJ = {}

exp_IDS = ['CJ']
model_name = ['z_v_prevrespconf','z_v_prevrespmetad']
for exp in exp_IDS:
    print(exp)
    models_reg_CJ[exp] = {}
    
    for mod_name in model_name:
        models_reg_CJ[exp][mod_name] = {}
        model_filename  = os.path.join(hddm_m_path, mod_name+'_'+exp)       
        file = open(model_filename+'_fm.db','rb')
        model_file = pickle.load(file)
        file.close()
        models_reg_CJ[exp][mod_name] = model_file  #hddm.load(model_filename+'_fm.db')
        

In [ ]:
models_reg_CJ[exp]['z_v_prevrespconf'].print_stats()

hddm_m_path = os.path.join(results_path,'Group_level_analyses','hddm_models')
models_stim = {}
exp = '3reps'
mod_name = 'stimcoding_diffprev'#'stimcoding_weights'
models_stim[exp] = {}
    
try:
    model_filename  = os.path.join(hddm_m_path, mod_name+'_'+exp)       
    file = open(model_filename+'_fm.db','rb')
    model_file = pickle.load(file)
except:
    print('exception')
file.close()
models_stim[exp][mod_name] = model_file  #hddm.load(model_filename+'_fm.db')

In [ ]:
coef = models_reg_CJ[exp]['z_v_prevrespmetad'].gen_stats()

In [ ]:
coef.loc[['v_G1sim:C(nrep)[0]:metad']]

In [ ]:
vint0 = coef.loc[['v_G1sim:C(nrep)[0]:metad','v_G2sim:C(nrep)[0]:metad', 'v_G3sim:C(nrep)[0]:metad','v_G4sim:C(nrep)[0]:metad','v_G5sim:C(nrep)[0]:metad','v_G6sim:C(nrep)[0]:metad'], ['mean','2.5q','97.5q']]
vint0['pos'] = np.array([1,2,3,4,5,6])
vint0['nrep'] = 0
vint0['param'] = 'v_metad'

vint1 = coef.loc[['v_G1sim:C(nrep)[1]:metad','v_G2sim:C(nrep)[1]:metad', 'v_G3sim:C(nrep)[1]:metad','v_G4sim:C(nrep)[1]:metad','v_G5sim:C(nrep)[1]:metad','v_G6sim:C(nrep)[1]:metad'], ['mean','2.5q','97.5q']]
vint1['pos'] = np.array([1,2,3,4,5,6])
vint1['nrep'] = 1
vint1['param'] = 'v_metad'

z = coef.loc[['z_response1:C(nrep)[0]:metad','z_response1:C(nrep)[0]:metad'], ['mean','2.5q','97.5q']]
z['nrep'] = np.array([0,1])
z['pos'] = 'N'
z['param'] = 'z_metad'

In [ ]:
 z['nrep'] = np.array([0,0,1,1])

In [ ]:
def get_reg_int_params(models, exp, mod_name):
    coef = models[exp][mod_name].gen_stats()
    if exp == 'CJ':
        if mod_name == 'z_v_prevrespconf': 
            vint0 = coef.loc[['v_G1sim:C(nrep)[0]:Confidence1','v_G2sim:C(nrep)[0]:Confidence1', 'v_G3sim:C(nrep)[0]:Confidence1','v_G4sim:C(nrep)[0]:Confidence1','v_G5sim:C(nrep)[0]:Confidence1','v_G6sim:C(nrep)[0]:Confidence1'], ['mean','2.5q','97.5q']]
            vint0['pos'] = np.array([1,2,3,4,5,6])
            vint0['nrep'] = 0
            vint0['param'] = 'v_confi'

            vint1 = coef.loc[['v_G1sim:C(nrep)[1]:Confidence1','v_G2sim:C(nrep)[1]:Confidence1', 'v_G3sim:C(nrep)[1]:Confidence1','v_G4sim:C(nrep)[1]:Confidence1','v_G5sim:C(nrep)[1]:Confidence1','v_G6sim:C(nrep)[1]:Confidence1'], ['mean','2.5q','97.5q']]
            vint1['pos'] = np.array([1,2,3,4,5,6])
            vint1['nrep'] = 1
            vint1['param'] = 'v_confi'

            z = coef.loc[['z_C(nrep)[0]:response1:Confidence1','z_C(nrep)[1]:response1:Confidence1'], ['mean','2.5q','97.5q']]
            base_val = z.iloc[0,0]
            z['nrep'] = np.array([0,1])
            z['pos'] = 'N'
            z['param'] = 'z_confi'

            pars = pd.concat([z,vint0,vint1])
            pars['model'] = mod_name
            pars['exp'] = exp
            
        if mod_name == 'z_v_prevrespmetad': 
            vint0 = coef.loc[['v_G1sim:C(nrep)[0]:metad','v_G2sim:C(nrep)[0]:metad', 'v_G3sim:C(nrep)[0]:metad','v_G4sim:C(nrep)[0]:metad','v_G5sim:C(nrep)[0]:metad','v_G6sim:C(nrep)[0]:metad'], ['mean','2.5q','97.5q']]
            vint0['pos'] = np.array([1,2,3,4,5,6])
            vint0['nrep'] = 0
            vint0['param'] = 'v_metad'

            vint1 = coef.loc[['v_G1sim:C(nrep)[1]:metad','v_G2sim:C(nrep)[1]:metad', 'v_G3sim:C(nrep)[1]:metad','v_G4sim:C(nrep)[1]:metad','v_G5sim:C(nrep)[1]:metad','v_G6sim:C(nrep)[1]:metad'], ['mean','2.5q','97.5q']]
            vint1['pos'] = np.array([1,2,3,4,5,6])
            vint1['nrep'] = 1
            vint1['param'] = 'v_metad'

            z = coef.loc[['z_response1:C(nrep)[0]:metad','z_response1:C(nrep)[1]:metad'], ['mean','2.5q','97.5q']]
            z['nrep'] = np.array([0,1])
            z['pos'] = 'N'
            z['param'] = 'z_metad'
            
            pars = pd.concat([z,vint0,vint1])
            pars['model'] = mod_name
            pars['exp'] = exp
            
            
        return(pars)

Concatenating models in dataframe

In [ ]:
i = 0
for im in models_reg_CJ:
    for mod_name in models_reg_CJ[im]:
        if i == 0:
            CJ_intparams_reg = get_reg_int_params(models_reg_CJ, im, mod_name)

            i = i + 1
        else:
            aux = get_reg_int_params(models_reg_CJ, im, mod_name)            
            CJ_intparams_reg = pd.concat([CJ_intparams_reg, aux])        
CJ_intparams_reg.reset_index(inplace = True, drop = True)     

plotting confidence interaction

In [ ]:

#estims = md_3reps_betas.copy()
fig, axes = plt.subplots(1, 2, figsize=(9,3.5), gridspec_kw={'width_ratios': [2, 1]})
fig.tight_layout(pad=2.0)

y_limits = np.array([[-1, 1] ,[-0.3, 0.0]])
estims = CJ_intparams_reg[(CJ_intparams_reg.exp == 'CJ' ) & (CJ_intparams_reg.model == 'z_v_prevrespconf')]

parameters = ['v_confi','z_confi']
nreps = [0,1]


for x, xparam in enumerate(parameters):
    for y, yrep in enumerate(nreps):
        if xparam == 'v_confi':
            stim = estims[(estims.param == xparam) & (estims.nrep == 0) ] # & (estims.nrep == yrep)
            sns.lineplot(data=stim, x="pos" , y="mean" , ax=axes[x],color= colpal2[0])
           
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs[0,:] = stim['mean'] - errs[0,:]  
            errs[1,:] = errs[1,:] - stim['mean']
            
            axes[x].errorbar(stim["pos"], stim["mean"], yerr=errs, fmt='none', ecolor=colpal2[0], elinewidth=1.5, capsize=5)

            stim = estims[(estims.param == xparam) & (estims.nrep == 1) ] # & (estims.nrep == yrep)
            sns.lineplot(data=stim, x="pos" , y="mean" , ax=axes[x],color= colpal2[1])
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords) 
            
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs[0,:] = stim['mean'] - errs[0,:]  
            errs[1,:] = errs[1,:] - stim['mean']
            
            axes[x].errorbar(stim["pos"], stim["mean"], yerr=errs, fmt='none', ecolor=colpal2[1], elinewidth=1.5, capsize=5)

   
            
            #axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            #fmt=' ', zorder=3, ecolor = 'black')

            axes[x].set_ylim(y_limits[0])
            axes[x].set_xlim([0.5,6.5])
            axes[x].axhline(0.0, ls='--', color= 'black', alpha = 0.2)
            
            axes[x].tick_params(axis='y', labelsize=15) 
            axes[x].tick_params(axis='x', labelsize=15)
            axes[x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[x].spines['top'].set_visible(False)
            axes[x].spines['right'].set_visible(False)
            axes[x].spines['left'].set_visible(False)
            
            axes[x].set_xlabel('Similarity N-1', fontsize = 15)
            axes[x].set_ylabel('v estimate (a.u.)', fontdict={'size':15}); 
            axes[x].margins(x=0.5)
            axes[x].set_xticks([1,2,3,4,5,6])

            axes[x].set_xticklabels(['1st','2nd','3rd','4th','5th','6th'])


    if xparam == 'z_confi':
        stim = estims[(estims.param == xparam)]
        g = sns.pointplot(data=stim, x="nrep" , y="mean" , hue="nrep",linestyles='', 
        dodge=True, join=False, ci=None,scale=1.5, palette = colpal2, errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[x])

        plt.setp(g.collections, alpha=.6) # set alpha for all points

        # Find the x,y coordinates for each point
        x_coords = []
        y_coords = []
        for point_pair in axes[x].collections:
            for xs, ys in point_pair.get_offsets():
                x_coords.append(xs)
                y_coords.append(ys)
        x_coords = np.ma.compressed(x_coords) # eliminate masked values
        y_coords = np.ma.compressed(y_coords)   

        errs = np.array(stim[["2.5q","97.5q"]]).T
        errs[0,:] = stim['mean'] - errs[0,:]  
        errs[1,:] = errs[1,:] - stim['mean']

        axes[x].errorbar(x_coords, y_coords, yerr=errs, #errs
        fmt=' ', zorder=3, ecolor = 'black')

        axes[x].errorbar(x_coords, y_coords, yerr=errs, #errs
        fmt=' ', zorder=3, ecolor = 'black')

        axes[x].set_ylim(y_limits[1])
        axes[x].set_xlim([-0.5,1.5])
        axes[x].tick_params(axis='y', labelsize=15) 
        axes[x].tick_params(axis='x', labelsize=15) 
        axes[x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
        axes[x].axhline(0.0, ls='--', color= 'black', alpha = 0.2)

        axes[x].spines['top'].set_visible(False)
        axes[x].spines['right'].set_visible(False)
        axes[x].spines['left'].set_visible(False)
        axes[x].set_xlabel('Choice N-1', fontsize = 15)
        axes[x].set_ylabel('z estimate (a.u.)', fontdict={'size':15}); 
        axes[x].margins(x=0.5)
        axes[x].set_xticklabels(['P1', 'P2'])
        axes[x].set_yticks([-0.2,0.0,0.2])
        axes[x].set_yticklabels([-0.2,0.0,0.2])

        #axes[y,x].margins(y=10)





figpath = os.path.join(figures_path, 'CJ_hddmREGxconfi.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)

        

In [ ]:

#estims = md_3reps_betas.copy()
fig, axes = plt.subplots(1, 2, figsize=(9,3.5), gridspec_kw={'width_ratios': [2, 1]})
fig.tight_layout(pad=2.0)

y_limits = np.array([[-1.5, 1.5] ,[-0.3, 0.0]])
estims = CJ_intparams_reg[(CJ_intparams_reg.exp == 'CJ' ) & (CJ_intparams_reg.model == 'z_v_prevrespmetad')]

parameters = ['v_metad','z_metad']
nreps = [0,1]


for x, xparam in enumerate(parameters):
    for y, yrep in enumerate(nreps):
        if xparam == 'v_metad':
            stim = estims[(estims.param == xparam) & (estims.nrep == 0) ] # & (estims.nrep == yrep)
            sns.lineplot(data=stim, x="pos" , y="mean" , ax=axes[x],color= colpal2[0])
           
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs[0,:] = stim['mean'] - errs[0,:]  
            errs[1,:] = errs[1,:] - stim['mean']
            
            axes[x].errorbar(stim["pos"], stim["mean"], yerr=errs, fmt='none', ecolor=colpal2[0], elinewidth=1.5, capsize=5)

            stim = estims[(estims.param == xparam) & (estims.nrep == 1) ] # & (estims.nrep == yrep)
            sns.lineplot(data=stim, x="pos" , y="mean" , ax=axes[x],color= colpal2[1])
            plt.setp(g.collections, alpha=.6) # set alpha for all points
            
            # Find the x,y coordinates for each point
            x_coords = []
            y_coords = []
            for point_pair in axes[x].collections:
                for xs, ys in point_pair.get_offsets():
                    x_coords.append(xs)
                    y_coords.append(ys)
            x_coords = np.ma.compressed(x_coords) # eliminate masked values
            y_coords = np.ma.compressed(y_coords) 
            
            errs = np.array(stim[["2.5q","97.5q"]]).T
            errs[0,:] = stim['mean'] - errs[0,:]  
            errs[1,:] = errs[1,:] - stim['mean']
            
            axes[x].errorbar(stim["pos"], stim["mean"], yerr=errs, fmt='none', ecolor=colpal2[1], elinewidth=1.5, capsize=5)

   
            
            #axes[y,x].errorbar(x_coords, y_coords, yerr=errs, #errs
            #fmt=' ', zorder=3, ecolor = 'black')

            axes[x].set_ylim(y_limits[0])
            axes[x].set_xlim([0.5,6.5])
            axes[x].axhline(0.0, ls='--', color= 'black', alpha = 0.2)
            
            axes[x].tick_params(axis='y', labelsize=15) 
            axes[x].tick_params(axis='x', labelsize=15)
            axes[x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
            
            axes[x].spines['top'].set_visible(False)
            axes[x].spines['right'].set_visible(False)
            axes[x].spines['left'].set_visible(False)
            
            axes[x].set_xlabel('Similarity N-1', fontsize = 15)
            axes[x].set_ylabel('v estimate (a.u.)', fontdict={'size':15}); 
            axes[x].margins(x=0.5)
            axes[x].set_xticks([1,2,3,4,5,6])

            axes[x].set_xticklabels(['1st','2nd','3rd','4th','5th','6th'])


    if xparam == 'z_metad':
        stim = estims[(estims.param == xparam)]
        g = sns.pointplot(data=stim, x="nrep" , y="mean" , hue="nrep",linestyles='', 
        dodge=True, join=False, ci=None,scale=1.5, palette = colpal2, errwidth=0, capsize=0,markers=['o', 'v', 's'], ax=axes[x])

        plt.setp(g.collections, alpha=.6) # set alpha for all points

        # Find the x,y coordinates for each point
        x_coords = []
        y_coords = []
        for point_pair in axes[x].collections:
            for xs, ys in point_pair.get_offsets():
                x_coords.append(xs)
                y_coords.append(ys)
        x_coords = np.ma.compressed(x_coords) # eliminate masked values
        y_coords = np.ma.compressed(y_coords)   

        errs = np.array(stim[["2.5q","97.5q"]]).T
        errs[0,:] = stim['mean'] - errs[0,:]  
        errs[1,:] = errs[1,:] - stim['mean']

        axes[x].errorbar(x_coords, y_coords, yerr=errs, #errs
        fmt=' ', zorder=3, ecolor = 'black')

        axes[x].errorbar(x_coords, y_coords, yerr=errs, #errs
        fmt=' ', zorder=3, ecolor = 'black')

        axes[x].set_ylim(y_limits[1])
        axes[x].set_xlim([-0.5,1.5])
        axes[x].tick_params(axis='y', labelsize=15) 
        axes[x].tick_params(axis='x', labelsize=15) 
        axes[x].legend(loc='lower left', borderpad=0.2,prop={'size':10}, fontsize=8,  framealpha= 0.0)
        axes[x].axhline(0.0, ls='--', color= 'black', alpha = 0.2)

        axes[x].spines['top'].set_visible(False)
        axes[x].spines['right'].set_visible(False)
        axes[x].spines['left'].set_visible(False)
        axes[x].set_xlabel('Choice N-1', fontsize = 15)
        axes[x].set_ylabel('z estimate (a.u.)', fontdict={'size':15}); 
        axes[x].margins(x=0.5)
        axes[x].set_xticklabels(['P1', 'P2'])
        axes[x].set_yticks([-0.2,0.0,0.2])
        axes[x].set_yticklabels([-0.2,0.0,0.2])

        #axes[y,x].margins(y=10)



figpath = os.path.join(figures_path, 'CJ_hddmREGxmetad.pdf')
plt.savefig(figpath ,bbox_inches='tight', dpi = 300)

         

In [ ]:
def get_reg_params(models, exp, mod_name):
    coef = models[exp][mod_name].gen_stats()
    if mod_name == 'stimcoding_diffprev':
        z = coef.loc[['z_Intercept','z_response1','z_C(nrep)[T.1]','z_response1:C(nrep)[T.1]'], ['mean','2.5q','97.5q']]
        base_val = z.iloc[0,0]
        z['nrep'] = np.array([0,0,1,1])
        z['prevr'] = np.array([0,1,0,1])
        z['pos'] = 'N'
        z['param'] = 'z'
        z['2.5q'] = z['2.5q'] - z['mean']  
        z['97.5q'] = z['97.5q'] - z['mean']  
        z['mean'] = z['mean']  + base_val
        z.iloc[0,0] = z.iloc[0,0] - base_val
        
        if exp != 'CJ':        
            z = coef.loc[['z_Intercept','z_response1','z_C(nrep)[T.1]','z_response1:C(nrep)[T.1]','z_C(nrep)[T.2]','z_response1:C(nrep)[T.2]'], ['mean','2.5q','97.5q']]
            base_val = z.iloc[0,0]
            z['nrep'] = np.array([0,0,1,1,2,2])
            z['prevr'] = np.array([0,1,0,1,0,1])
            z['pos'] = 'N'
            z['param'] = 'z'
            z['2.5q'] = z['2.5q'] - z['mean']  
            z['97.5q'] = z['97.5q'] - z['mean']  
            z['mean'] = z['mean']  + base_val
            z.iloc[0,0] = z.iloc[0,0] - base_val


        #v_Intercept
        base_val = coef.loc['v_Intercept']['mean']

        v0 = coef.loc[['v_G1sim:C(nrep)[0]','v_G2sim:C(nrep)[0]','v_G3sim:C(nrep)[0]','v_G4sim:C(nrep)[0]','v_G5sim:C(nrep)[0]','v_G6sim:C(nrep)[0]'], ['mean','2.5q','97.5q']]
        v0['nrep'] = 0
        v0['prevr'] = 'N'
        v0['pos'] = np.array([1,2,3,4,5,6])
        v0['param'] = 'v'
        v0['2.5q'] = v0['2.5q'] - v0['mean']  
        v0['97.5q'] = v0['97.5q'] - v0['mean']  
        v0['mean']  = v0['mean']  + base_val 

        v1 = coef.loc[['v_G1sim:C(nrep)[1]','v_G2sim:C(nrep)[1]','v_G3sim:C(nrep)[1]','v_G4sim:C(nrep)[1]','v_G5sim:C(nrep)[1]','v_G6sim:C(nrep)[1]'], ['mean','2.5q','97.5q']]
        v1['nrep'] = 1
        v1['prevr'] = 'N'
        v1['pos'] = np.array([1,2,3,4,5,6])
        v1['param'] = 'v'
        v1['2.5q'] = v1['2.5q'] - v1['mean']  
        v1['97.5q'] = v1['97.5q'] - v1['mean']
        v1['mean']  = v1['mean']  + base_val 
        pars = pd.concat([z,v0,v1])
        pars['model'] = mod_name
        pars['exp'] = exp

        if exp != 'CJ':

            v2 = coef.loc[['v_G1sim:C(nrep)[2]','v_G2sim:C(nrep)[2]','v_G3sim:C(nrep)[2]','v_G4sim:C(nrep)[2]','v_G5sim:C(nrep)[2]','v_G6sim:C(nrep)[2]'], ['mean','2.5q','97.5q']]
            v2['nrep'] = 2
            v2['prevr'] = 'N'
            v2['pos'] = np.array([1,2,3,4,5,6])
            v2['param'] = 'v'
            v2['2.5q'] = v2['2.5q'] - v2['mean']  
            v2['97.5q'] = v2['97.5q'] - v2['mean']
            v2['mean']  = v2['mean']  + base_val 
            pars = pd.concat([z,v0,v1,v2])
            pars['model'] = mod_name
            pars['exp'] = exp
        return(pars)




In [ ]:
def get_reg_params(models, exp, mod_name):
    coef = models[exp][mod_name].gen_stats()
    if mod_name == 'stimcoding_diffprev':
        z = coef.loc[['z_Intercept','z_response1','z_C(nrep)[T.1]','z_response1:C(nrep)[T.1]'], ['mean','2.5q','97.5q']]
        base_val = z.iloc[0,0]
        z['nrep'] = np.array([0,0,1,1])
        z['prevr'] = np.array([0,1,0,1])
        z['pos'] = 'N'
        z['param'] = 'z'
        z['2.5q'] = z['2.5q'] - z['mean']  
        z['97.5q'] = z['97.5q'] - z['mean']  
        z['mean'] = z['mean']  + base_val
        z.iloc[0,0] = z.iloc[0,0] - base_val
        
        if exp != 'CJ':        
            z = coef.loc[['z_Intercept','z_response1','z_C(nrep)[T.1]','z_response1:C(nrep)[T.1]','z_C(nrep)[T.2]','z_response1:C(nrep)[T.2]'], ['mean','2.5q','97.5q']]
            base_val = z.iloc[0,0]
            z['nrep'] = np.array([0,0,1,1,2,2])
            z['prevr'] = np.array([0,1,0,1,0,1])
            z['pos'] = 'N'
            z['param'] = 'z'
            z['2.5q'] = z['2.5q'] - z['mean']  
            z['97.5q'] = z['97.5q'] - z['mean']  
            z['mean'] = z['mean']  + base_val
            z.iloc[0,0] = z.iloc[0,0] - base_val


        #v_Intercept
        base_val = coef.loc['v_Intercept']['mean']

        v0 = coef.loc[['v_G1sim:C(nrep)[0]','v_G2sim:C(nrep)[0]','v_G3sim:C(nrep)[0]','v_G4sim:C(nrep)[0]','v_G5sim:C(nrep)[0]','v_G6sim:C(nrep)[0]'], ['mean','2.5q','97.5q']]
        v0['nrep'] = 0
        v0['prevr'] = 'N'
        v0['pos'] = np.array([1,2,3,4,5,6])
        v0['param'] = 'v'
        v0['2.5q'] = v0['2.5q'] - v0['mean']  
        v0['97.5q'] = v0['97.5q'] - v0['mean']  
        v0['mean']  = v0['mean']  + base_val 

        v1 = coef.loc[['v_G1sim:C(nrep)[1]','v_G2sim:C(nrep)[1]','v_G3sim:C(nrep)[1]','v_G4sim:C(nrep)[1]','v_G5sim:C(nrep)[1]','v_G6sim:C(nrep)[1]'], ['mean','2.5q','97.5q']]
        v1['nrep'] = 1
        v1['prevr'] = 'N'
        v1['pos'] = np.array([1,2,3,4,5,6])
        v1['param'] = 'v'
        v1['2.5q'] = v1['2.5q'] - v1['mean']  
        v1['97.5q'] = v1['97.5q'] - v1['mean']
        v1['mean']  = v1['mean']  + base_val 
        pars = pd.concat([z,v0,v1])
        pars['model'] = mod_name
        pars['exp'] = exp

        if exp != 'CJ':

            v2 = coef.loc[['v_G1sim:C(nrep)[2]','v_G2sim:C(nrep)[2]','v_G3sim:C(nrep)[2]','v_G4sim:C(nrep)[2]','v_G5sim:C(nrep)[2]','v_G6sim:C(nrep)[2]'], ['mean','2.5q','97.5q']]
            v2['nrep'] = 2
            v2['prevr'] = 'N'
            v2['pos'] = np.array([1,2,3,4,5,6])
            v2['param'] = 'v'
            v2['2.5q'] = v2['2.5q'] - v2['mean']  
            v2['97.5q'] = v2['97.5q'] - v2['mean']
            v2['mean']  = v2['mean']  + base_val 
            pars = pd.concat([z,v0,v1,v2])
            pars['model'] = mod_name
            pars['exp'] = exp
        return(pars)


